# TradeSense · Component 03 · notebook 40: the Watch state and the volume-burst question (CPU, no training)

**Why.** Abnormal volume bursts are weak for every model (fusion 0.18, union 0.28 at the 2/day alert rule). A natural volume spike and a manipulated one often look the same on 1-minute bars. This notebook does two things, using only the saved fusion v0 scores (no new fitting):

1. **Ambiguity diagnostic (all eight families).** For every CAL-3 event, the peak fused score inside the event is ranked against the daily maxima of the same fused score on the genuine August series. `rank = share of genuine days whose top bar is below the event peak`. A family is called **distinguishable** if its median rank is >= 0.80, **ambiguous at 1-minute level** if < 0.50, otherwise **partly distinguishable**. (Written before the run.)
2. **Three-level state: Normal / Watch / Alert.** *Alert* = the frozen fusion rule (T1c, 2 episodes/day, unchanged). *Watch* = the same fused score with a lower rolling threshold at **4 and 8 episodes/day** (thresholds fixed from the out-of-fold genuine DEV bars exactly as in notebook 35; levels declared here, before the run). Watch is shown to the user as "unusual, keep an eye on it", never as an alert.

**Pre-registered hypotheses (CAL-3, fusion v0, T1c).**
- **H-W0 (integrity):** the Alert recall recomputed here equals notebook 35 (macro 8 at 2/day). The frozen alert rule is not touched.
- **H-W1 (Watch helps volume bursts):** at Watch-8 the recall of abnormal_volume_burst rises by >= 0.15 over Alert, paired 95% bootstrap CI above 0, and the genuine Watch episodes/day stay <= 10 (1.25 x 8).
- **H-W2 (Watch is not a dump):** the macro-8 gain of Watch-8 over Alert is reported with the genuine episodes/day it costs; no family is allowed to drop (Watch is a superset of Alert by construction).
- If H-W1 fails, the honest reading is that volume bursts need information that 1-minute bars do not carry (trade-level phase 2: who traded, order sizes, the order book), and the report says so.

TEST-3, TEST-B and every TGNN test day are untouched. Nothing here changes a frozen rule.

In [ ]:
import os, sys, json, time, hashlib, warnings, datetime
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
for f in ['declared_fusion_v0.json', 'fusion_v0_cal3_table.csv']:
    assert (OUT/f).exists(), f'{f} missing: run notebook 35 (the patched one, with the A6 autoencoder) to the end first'
for s in SYMBOLS: assert (OUT/'scores'/f'fusion_v0_{s}.npz').exists(), 'run notebook 35 first'
sys.path.insert(0, str(OUT)); import importlib; import b2_lib as B; importlib.reload(B)
say('protocol B2 intact, sha256 ' + ph[:16] + '; library sha256 ' + sha_file(OUT/'b2_lib.py') + ('  [MOCK RUN]' if MOCK else ''))
WATCH_BUDGETS = [4, 8]; ALERT_BUDGET = B.PRIMARY; BUDS = [ALERT_BUDGET] + WATCH_BUDGETS
RANK_DIST, RANK_AMB = 0.80, 0.50; H_W1_GAIN = 0.15; H_W1_EPS = 1.25 * 8

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944


## 1 · Data (saved fusion v0 scores, CAL-3 events, thresholds)

In [ ]:
AOLEN = 31 * 1440; CAL_LO = 1440; DEV_LO, DEV_HI = 90720, 129600
CALV = B.valid_mask(AOLEN, CAL_LO, AOLEN)
VD = np.zeros(129600, bool); VD[DEV_LO + 1440:DEV_HI] = True; VD[:B.WARM] = False
FS = {}; EV = {}
for s in SYMBOLS:
    z = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); assert not any('TEST' in k for k in z.files)
    FS[s] = {k: z[k] for k in ['GEN_AO', 'CAL_r0', 'OOF_GEN_L1']}
    assert len(FS[s]['GEN_AO']) == AOLEN and len(FS[s]['CAL_r0']) == AOLEN and len(FS[s]['OOF_GEN_L1']) == 129600
    e = pd.read_csv(L3/f'l3v3_{s}_CAL_events.csv'); e = e[e.replica == 0].reset_index(drop=True); assert e.t0.min() >= CAL_LO and (e.t0 + e.dur).max() <= AOLEN; EV[s] = e
FAMN = B.FAMS; FAMS = list(FAMN)
EALL = pd.concat([EV[s].assign(sym=s) for s in SYMBOLS], ignore_index=True)
say(f'fusion v0 scores loaded; CAL-3 events {len(EALL)} (per family {EALL.family.value_counts().sort_index().to_dict()})')
def thr_roll(g, q): t = B.rolling_thr(g, q); return np.where(np.isnan(t), np.inf, t)
def qsel(g, bud):
    for q in B.QGRID:
        if B.eps_per_day(g > thr_roll(g, q), VD) <= bud: return q
    return B.QGRID[-1]
def event_hits(a, ev):
    c = np.concatenate([[0], np.cumsum(np.nan_to_num(a).astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
QS = {b: {s: qsel(FS[s]['OOF_GEN_L1'], b) for s in SYMBOLS} for b in BUDS}
HIT = {}; GENEPS = {}; ALARM = {}
for b in BUDS:
    gens = []; hs = {}
    for s in SYMBOLS:
        thr = thr_roll(FS[s]['GEN_AO'], QS[b][s]); gen = FS[s]['GEN_AO'] > thr; cal = FS[s]['CAL_r0'] > thr; ALARM[(b, s)] = cal; hs[s] = event_hits(cal, EV[s]); gens.append(B.eps_per_day(gen, CALV))
    HIT[b] = np.concatenate([hs[s] for s in SYMBOLS]); GENEPS[b] = float(np.mean(gens))
say('rolling-threshold quantile per budget (from the out-of-fold genuine DEV fused score): ' + str({b: {s: QS[b][s] for s in SYMBOLS} for b in BUDS}))

fusion v0 scores loaded; CAL-3 events 400 (per family {1: 50, 2: 50, 3: 50, 4: 50, 5: 50, 6: 50, 7: 50, 8: 50})
rolling-threshold quantile per budget (from the out-of-fold genuine DEV fused score): {2: {'BTCUSDT': 0.997, 'ETHUSDT': 0.997}, 4: {'BTCUSDT': 0.993, 'ETHUSDT': 0.995}, 8: {'BTCUSDT': 0.99, 'ETHUSDT': 0.99}}


## 2 · Unit tests

In [ ]:
def macro8_of(h):
    d = pd.DataFrame({'sym': EALL.sym, 'fam': EALL.family, 'h': h}); return float(d.groupby(['sym', 'fam']).h.mean().groupby('fam').mean().mean())
# T01 the Alert here equals notebook 35 (fusion, budget 2, macro 8)
n35 = pd.read_csv(OUT/'fusion_v0_cal3_table.csv'); ref = float(n35[(n35.method == 'fusion') & (n35.budget == ALERT_BUDGET)]['macro 8'].iloc[0])
assert abs(macro8_of(HIT[ALERT_BUDGET]) - ref) < 5e-4, (macro8_of(HIT[ALERT_BUDGET]), ref); say(f'PASS T01 Alert recall recomputed here equals notebook 35 (macro 8 {ref:.3f}); the frozen alert is untouched')
# T02 hit rule equals a plain loop
a = np.zeros(200, bool); a[[10, 50, 51]] = True; ev_ = pd.DataFrame({'t0': [5, 40, 60, 0], 'dur': [4, 5, 10, 11]}); plain = np.array([a[t:t + d].any() for t, d in zip(ev_.t0, ev_.dur)])
assert (event_hits(a, ev_) == plain).all(); say('PASS T02 event hit rule equals a plain loop')
# T03 Watch is a superset of Alert (lower threshold => every Alert bar is also a Watch bar) and the quantile order is right
for s in SYMBOLS:
    assert QS[8][s] <= QS[4][s] <= QS[2][s]
    for b in WATCH_BUDGETS: assert not (ALARM[(ALERT_BUDGET, s)] & ~ALARM[(b, s)]).any()
assert all((HIT[b] >= HIT[ALERT_BUDGET]).all() for b in WATCH_BUDGETS); say('PASS T03 Watch alarms contain every Alert alarm (no family can drop)')
# T04 rank function
def rank_of(p, daily): return float(np.mean(np.asarray(daily) < p))
assert rank_of(5, [1, 2, 3, 4]) == 1.0 and rank_of(0, [1, 2, 3, 4]) == 0.0 and rank_of(2.5, [1, 2, 3, 4]) == 0.5; say('PASS T04 rank function (share of genuine days below the event peak)')
say('PASS T05 only CAL-3 and DEV out-of-fold series are read (no TEST key in the score files, TEST files are not opened)')

PASS T01 Alert recall recomputed here equals notebook 35 (macro 8 0.778); the frozen alert is untouched
PASS T02 event hit rule equals a plain loop
PASS T03 Watch alarms contain every Alert alarm (no family can drop)
PASS T04 rank function (share of genuine days below the event peak)
PASS T05 only CAL-3 and DEV out-of-fold series are read (no TEST key in the score files, TEST files are not opened)


## 3 · Ambiguity diagnostic

In [ ]:
def daily_max(g):
    out = []
    for i in range(CAL_LO, AOLEN - 1439, 1440):
        w = g[i:i + 1440]
        if np.isfinite(w).any(): out.append(np.nanmax(w))
    return np.array(out)
DM = {s: daily_max(FS[s]['GEN_AO']) for s in SYMBOLS}
rk = []
for s in SYMBOLS:
    for r in EV[s].itertuples():
        w = FS[s]['CAL_r0'][r.t0:r.t0 + r.dur]; p = np.nanmax(w) if np.isfinite(w).any() else -np.inf
        rk.append({'sym': s, 'fam': r.family, 'amp': r.amp, 'rank': rank_of(p, DM[s])})
RK = pd.DataFrame(rk); AMB = RK.groupby('fam').agg(median_rank=('rank', 'median'), share_above_all_genuine_days=('rank', lambda x: float((x >= 1.0).mean()))).reset_index()
AMB['family'] = AMB.fam.map(FAMN); AMB['reading'] = np.where(AMB.median_rank >= RANK_DIST, 'distinguishable', np.where(AMB.median_rank < RANK_AMB, 'ambiguous at 1-minute level', 'partly distinguishable'))
RK['tercile'] = RK.groupby('fam').amp.transform(lambda x: pd.qcut(x.rank(method='first'), 3, labels=['low amp', 'mid amp', 'high amp']))
AT = RK.pivot_table(index='fam', columns='tercile', values='rank', aggfunc='median').reset_index(); AT['family'] = AT.fam.map(FAMN)
say(''); say(f'--- ambiguity: event peak fused score ranked against the {len(DM[SYMBOLS[0]])} genuine August daily maxima (per symbol) ---'); say(AMB[['family', 'median_rank', 'share_above_all_genuine_days', 'reading']].round(2).to_string(index=False))
say(''); say('median rank by amplitude tercile (inside each family)'); say(AT[['family', 'low amp', 'mid amp', 'high amp']].round(2).to_string(index=False))


--- ambiguity: event peak fused score ranked against the 30 genuine August daily maxima (per symbol) ---
                   family  median_rank  share_above_all_genuine_days                     reading
       pump_and_dump_like         0.97                          0.24             distinguishable
       fake_breakout_like         0.97                          0.28             distinguishable
    abnormal_volume_burst         0.00                          0.00 ambiguous at 1-minute level
     trade_flow_imbalance         0.97                          0.38             distinguishable
wash_like_aggregate_proxy         1.00                          0.82             distinguishable
   liquidity_stress_proxy         0.97                          0.36             distinguishable
   momentum_ignition_like         0.88                          0.22             distinguishable
   slow_accumulation_like         0.90                          0.14             distinguishable

median rank by ampli

## 4 · Watch tiers

In [ ]:
rg = np.random.default_rng(0); FAMIDX = {f: np.flatnonzero((EALL.family == f).to_numpy()) for f in FAMS}
def boot_fam(h_a, h_b, f, n=NBOOT):
    ix = FAMIDX[f]; draws = rg.integers(0, len(ix), size=(n, len(ix))); a = h_a[ix][draws].mean(axis=1); b = h_b[ix][draws].mean(axis=1); return a, b, b - a
rows = []
for f in FAMS:
    ix = FAMIDX[f]; row = {'family': FAMN[f]}
    for b in BUDS:
        row[f'recall {"Alert" if b == ALERT_BUDGET else "Watch-" + str(b)}'] = float(HIT[b][ix].mean())
    _, wb, gain = boot_fam(HIT[ALERT_BUDGET], HIT[8], f); row['Watch-8 CI'] = f'{np.percentile(wb, 2.5):.2f}-{np.percentile(wb, 97.5):.2f}'; row['gain 8 vs Alert'] = float(HIT[8][ix].mean() - HIT[ALERT_BUDGET][ix].mean()); row['gain CI'] = f'{np.percentile(gain, 2.5):+.2f}..{np.percentile(gain, 97.5):+.2f}'; row['_lo'] = float(np.percentile(gain, 2.5)); rows.append(row)
WT = pd.DataFrame(rows)
mac = {b: macro8_of(HIT[b]) for b in BUDS}
say(''); say('--- CAL-3 recall per family: Alert (2/day, frozen) versus Watch tiers (fusion v0, T1c) ---'); say(WT.drop(columns='_lo').round(2).to_string(index=False))
say(''); say('--- macro 8 and the genuine cost (CAL-3 genuine August bars) ---')
CT = pd.DataFrame({'tier': ['Alert (2/day)', 'Watch-4', 'Watch-8'], 'genuine eps/day': [GENEPS[b] for b in BUDS], 'macro 8': [mac[b] for b in BUDS]}); say(CT.round(3).to_string(index=False))
vb = WT[WT.family == 'abnormal_volume_burst'].iloc[0]; HW1 = bool(vb['gain 8 vs Alert'] >= H_W1_GAIN and vb['_lo'] > 0 and GENEPS[8] <= H_W1_EPS)
say(''); say(f"H-W1 (volume burst, Watch-8): gain {vb['gain 8 vs Alert']:+.2f} (95% CI {vb['gain CI']}), genuine Watch-8 episodes/day {GENEPS[8]:.2f} (limit {H_W1_EPS:.1f}) -> {HW1}")
say(f'H-W2 macro-8 gain Watch-8 over Alert: {mac[8] - mac[ALERT_BUDGET]:+.3f} for {GENEPS[8] - GENEPS[ALERT_BUDGET]:+.2f} more genuine episodes/day; Watch-4: {mac[4] - mac[ALERT_BUDGET]:+.3f} for {GENEPS[4] - GENEPS[ALERT_BUDGET]:+.2f}')


--- CAL-3 recall per family: Alert (2/day, frozen) versus Watch tiers (fusion v0, T1c) ---
                   family  recall Alert  recall Watch-4  recall Watch-8 Watch-8 CI  gain 8 vs Alert      gain CI
       pump_and_dump_like          0.84            0.86            0.92  0.84-0.98             0.08 +0.02..+0.16
       fake_breakout_like          0.84            0.88            0.88  0.78-0.96             0.04 +0.00..+0.10
    abnormal_volume_burst          0.18            0.24            0.28  0.16-0.42             0.10 +0.02..+0.18
     trade_flow_imbalance          0.84            0.88            0.88  0.78-0.96             0.04 +0.00..+0.10
wash_like_aggregate_proxy          1.00            1.00            1.00  1.00-1.00             0.00 +0.00..+0.00
   liquidity_stress_proxy          0.96            0.98            1.00  1.00-1.00             0.04 +0.00..+0.10
   momentum_ignition_like          0.78            0.78            0.84  0.74-0.94             0.06 +0.00..+0.14
   s

## 5 · Reading

In [ ]:
vbr = AMB[AMB.family == 'abnormal_volume_burst'].iloc[0]
say(''); say('--- reading (pre-registered rules) ---')
say(f"abnormal_volume_burst is {vbr.reading} (median rank {vbr.median_rank:.2f}; rule: >= {RANK_DIST} distinguishable, < {RANK_AMB} ambiguous).")
if HW1: say('Policy for the product: volume-burst-like behaviour is shown as WATCH (amber), never as an alert on its own; the Watch state recovers a meaningful share of these events at a cost reported above.')
else: say('Watch does not recover volume bursts at an acceptable cost: they stay a documented limit of 1-minute bars. The route is trade-level phase 2 (order sizes, who traded, book changes), not a lower threshold.')
amb = AMB[AMB.reading != 'distinguishable'].family.tolist(); say('families that are not fully distinguishable from a normal day at bar level: ' + (', '.join(amb) if amb else 'none'))


--- reading (pre-registered rules) ---
abnormal_volume_burst is ambiguous at 1-minute level (median rank 0.00; rule: >= 0.8 distinguishable, < 0.5 ambiguous).
Watch does not recover volume bursts at an acceptable cost: they stay a documented limit of 1-minute bars. The route is trade-level phase 2 (order sizes, who traded, book changes), not a lower threshold.
families that are not fully distinguishable from a normal day at bar level: abnormal_volume_burst


## 6 · Save and declare

In [ ]:
WT.drop(columns='_lo').to_csv(OUT/'watch_tiers_cal3.csv', index=False); AMB.drop(columns='fam').to_csv(OUT/'ambiguity_cal3.csv', index=False); CT.to_csv(OUT/'watch_cost_cal3.csv', index=False)
DECLW = {'status': 'DECLARED before TEST-3 / TEST-B: Normal / Watch / Alert states on the fusion v0 score; the Alert rule is the frozen one.', 'declared_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z', 'score': 'fusion_v0 (declared_fusion_v0.json)',
         'alert': {'rule': 'T1c', 'budget_eps_per_day': ALERT_BUDGET, 'q': QS[ALERT_BUDGET]}, 'watch': {'rule': 'T1c', 'budgets_eps_per_day': WATCH_BUDGETS, 'q': {str(b): QS[b] for b in WATCH_BUDGETS}, 'display': 'amber, never an alert'},
         'cal3': {'genuine_eps_day': {str(b): GENEPS[b] for b in BUDS}, 'macro8': {str(b): mac[b] for b in BUDS}, 'H_W1_volume_burst_holds': HW1}, 'fusion_declared_sha16': sha_file(OUT/'declared_fusion_v0.json')}
(OUT/'declared_watch_v1.json').write_text(json.dumps(DECLW, indent=1, default=str))
block = '\n'.join(LINES); (OUT/'b2_step12_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step12_share_block.txt, or copy everything below) ====='); print(block); print('saved', OUT/'b2_step12_share_block.txt')


===== SHARE BLOCK (attach models_b2_v1/b2_step12_share_block.txt, or copy everything below) =====
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944
fusion v0 scores loaded; CAL-3 events 400 (per family {1: 50, 2: 50, 3: 50, 4: 50, 5: 50, 6: 50, 7: 50, 8: 50})
rolling-threshold quantile per budget (from the out-of-fold genuine DEV fused score): {2: {'BTCUSDT': 0.997, 'ETHUSDT': 0.997}, 4: {'BTCUSDT': 0.993, 'ETHUSDT': 0.995}, 8: {'BTCUSDT': 0.99, 'ETHUSDT': 0.99}}
PASS T01 Alert recall recomputed here equals notebook 35 (macro 8 0.778); the frozen alert is untouched
PASS T02 event hit rule equals a plain loop
PASS T03 Watch alarms contain every Alert alarm (no family can drop)
PASS T04 rank function (share of genuine days below the event peak)
PASS T05 only CAL-3 and DEV out-of-fold series are read (no TEST key in the score files, TEST files are not opened)

--- ambiguity: event peak fused score ranked against the 30 genuine August daily maxima (per symbol) -